# Kaismart Solutions S.A.S. — Extracción y EDA inicial

**Objetivo:** extraer las dos fuentes de información de Kaismart (sistema comercial en MySQL y sistema logístico en Excel), y realizar una exploración inicial de su estructura, calidad y contenido, siguiendo el enunciado de la evaluación (Partes 1 a 6).

- Fuente 1 — `df_ventas`: tabla `ventas` de la base de datos MySQL `clientes` (5.000 registros esperados).
- Fuente 2 — `df_logistica`: archivo `kaismart_eventos_logisticos.xlsx` (50.000 registros esperados).
- Llave de integración entre ambas fuentes: `pedido_id` (se usará más adelante, en el notebook de transformación Medallion).

> Nota de uso: si aún no cuenta con la base de datos MySQL ni el archivo Excel reales, ejecute `python scripts/generate_mock_data.py` desde la raíz del proyecto para generar datos simulados con problemas de calidad realistas y poder correr este notebook de punta a punta.

In [ ]:
import sys
from pathlib import Path

# Permite importar el paquete kaismart_etl (carpeta src/) sin instalarlo
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd

from kaismart_etl import business_questions as bq
from kaismart_etl import quality, stats
from kaismart_etl.extract import extract_logistica, extract_ventas, mostrar_comprobacion

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

## Parte 1. Extracción desde la base de datos (MySQL)

Se importan las librerías necesarias (`sqlalchemy`, `pymysql`, `pandas`), se crea la conexión con la base de datos `clientes`, se consulta **toda** la tabla `ventas` y se cierra la conexión correctamente (ver `kaismart_etl/db.py` y `kaismart_etl/extract.py`, que usan un bloque `try/finally` para garantizar el cierre incluso si ocurre un error).

In [ ]:
# extract_ventas() abre la conexion, ejecuta "SELECT * FROM ventas" y CIERRA
# la conexion en un bloque finally (ver kaismart_etl/extract.py)
df_ventas = extract_ventas()

mostrar_comprobacion(df_ventas, "df_ventas")

## Parte 2. Extracción desde el archivo Excel

Se lee `kaismart_eventos_logisticos.xlsx` con `pandas.read_excel` (motor `openpyxl`) y se almacena en `df_logistica`.

In [ ]:
df_logistica = extract_logistica()

mostrar_comprobacion(df_logistica, "df_logistica")

## Parte 3. Comprensión inicial de los datasets

**¿Qué representa una fila de `df_ventas`?** Cada fila es **una transacción de venta**: un cliente compra una cantidad de un producto de una categoría determinada, en una ciudad y a través de un canal (tienda física, web o app), en una fecha específica, con un precio, un descuento y una calificación opcional del cliente.

**¿Qué representa una fila de `df_logistica`?** Cada fila es **un evento del ciclo de vida logístico de un pedido** (por ejemplo: "Pedido Recibido", "Despachado", "Entregado"). Por lo tanto, **un mismo `pedido_id` tiene múltiples filas** en `df_logistica` (una por cada etapa), lo que explica por qué esta fuente tiene ~10 veces más registros (50.000) que `df_ventas` (5.000).

A continuación se determina, para cada dataset: número de registros y variables, nombre y tipo de cada variable, no nulos, e identificación heurística de identificadores / categóricas / numéricas / fechas.

In [ ]:
print(f"df_ventas: {df_ventas.shape[0]} registros, {df_ventas.shape[1]} variables")
print(f"df_logistica: {df_logistica.shape[0]} registros, {df_logistica.shape[1]} variables\n")

resumen_ventas_estructura = quality.resumen_estructura(df_ventas, "df_ventas")
resumen_logistica_estructura = quality.resumen_estructura(df_logistica, "df_logistica")

print("Estructura df_ventas:")
display(resumen_ventas_estructura)
print("\nEstructura df_logistica:")
display(resumen_logistica_estructura)

In [ ]:
# Clasificacion heuristica de variables: identificadores / categoricas / numericas / fechas
clasificacion_ventas = quality.sugerir_tipos_variables(df_ventas)
clasificacion_logistica = quality.sugerir_tipos_variables(df_logistica)

print("df_ventas ->", clasificacion_ventas)
print("df_logistica ->", clasificacion_logistica)

# Rango de fechas disponible en cada fuente
if "fecha_venta" in df_ventas.columns:
    print("\nRango fechas df_ventas:", quality.rango_fechas(df_ventas, "fecha_venta"))
if "fecha_evento" in df_logistica.columns:
    print("Rango fechas df_logistica:", quality.rango_fechas(df_logistica, "fecha_evento"))

**Otros aspectos de calidad observados (punto 10):**

- `ciudad`, `canal`, `categoria` (ventas) y `estado_evento`, `ciudad_destino`, `transportadora` (logística) son texto libre con **inconsistencias de mayúsculas/minúsculas, tildes y espacios** (p. ej. "bogota", "BOGOTÁ", "Bogotá D.C. ") en lugar de un catálogo controlado.
- `costo_envio` llega mezclado: algunos valores numéricos y otros como texto con formato moneda (`"$ 45.000"`), lo que impide operar la columna directamente.
- `id_venta`, `pedido_id` y `evento_id` deberían ser claves únicas, pero se detectan duplicados (ver Parte 4.3).
- `pedido_id` es la única llave común entre las dos fuentes; algunos registros la tienen nula, lo que impedirá integrarlos en la capa Gold.

## Parte 4. Perfil inicial de calidad del dato

### 4.1 Valores nulos

Se cuantifica cantidad y porcentaje de nulos por variable (sin eliminar ni imputar nada), ordenado de mayor a menor.

**Lectura esperada:**
- `calificacion_cliente` con alto % de nulos es **normal**: el cliente no está obligado a calificar la compra.
- `incidencia` con alto % de nulos en logística es **normal**: la mayoría de los eventos no tiene ninguna incidencia asociada.
- `valor_descuento` nulo también es esperable cuando no se aplicó descuento, aunque idealmente debería llegar como `0` desde el origen (posible problema de captura).
- Nulos en `pedido_id`, `fecha_venta`/`fecha_evento`, `precio_unitario` o `costo_envio` sí representan un **problema de calidad**, porque afectan la trazabilidad, el cálculo financiero o la integración entre fuentes.

In [ ]:
print("Perfil de nulos - df_ventas")
display(quality.perfil_nulos(df_ventas))

print("\nPerfil de nulos - df_logistica")
display(quality.perfil_nulos(df_logistica))

### 4.2 Valores únicos (cardinalidad)

Se usa `nunique()` para clasificar cada variable como de alta o baja cardinalidad, y se revisan específicamente `id_venta`, `pedido_id` y `evento_id` (deberían comportarse como identificadores, es decir, cardinalidad ≈ número de filas).

In [ ]:
print("Cardinalidad - df_ventas")
display(quality.perfil_cardinalidad(df_ventas))

print("\nCardinalidad - df_logistica")
display(quality.perfil_cardinalidad(df_logistica))

print("\nAtencion a identificadores:")
for col in ("id_venta", "pedido_id"):
    if col in df_ventas.columns:
        print(f"  df_ventas.{col}: {df_ventas[col].nunique()} valores unicos / {len(df_ventas)} filas")
for col in ("evento_id", "pedido_id"):
    if col in df_logistica.columns:
        print(f"  df_logistica.{col}: {df_logistica[col].nunique()} valores unicos / {len(df_logistica)} filas")

### 4.3 Duplicados

Se cuantifican filas completamente duplicadas y también duplicados en las columnas que deberían ser identificadores únicos (`id_venta`, `pedido_id` en ventas; `evento_id` en logística). **No se elimina nada en esta parte**, solo se documenta.

In [ ]:
print("Duplicados - df_ventas")
print(quality.perfil_duplicados(df_ventas, columnas_id=["id_venta", "pedido_id"]))

print("\nDuplicados - df_logistica")
print(quality.perfil_duplicados(df_logistica, columnas_id=["evento_id", "pedido_id"]))

**Interpretación importante:** en `df_logistica`, `pedido_id` se repite de forma **legítima y esperada** (cada pedido tiene varios eventos/etapas), por lo que no debe tratarse como un problema de calidad. El identificador que sí debe ser único en logística es `evento_id`. En `df_ventas`, tanto `id_venta` como `pedido_id` deberían ser únicos (una fila = una venta = un pedido); cualquier duplicado ahí sí es un error de origen a documentar y corregir en Silver.

### 4.4 Tipos de datos

Se revisa si el tipo detectado por pandas es coherente con el significado real de cada variable (fechas, identificadores, monetarias, numéricas, categóricas).

In [ ]:
tipos_esperados_ventas = {
    "id_venta": "identificador",
    "pedido_id": "identificador",
    "cliente_id": "identificador",
    "fecha_venta": "fecha",
    "ciudad": "categorica",
    "canal": "categorica",
    "categoria": "categorica",
    "metodo_pago": "categorica",
    "cantidad": "numerica",
    "precio_unitario": "monetaria",
    "valor_bruto": "monetaria",
    "valor_descuento": "monetaria",
    "valor_neto": "monetaria",
    "calificacion_cliente": "numerica",
}
print("Revision de tipos - df_ventas")
display(quality.revisar_tipos(df_ventas, tipos_esperados_ventas))

tipos_esperados_logistica = {
    "evento_id": "identificador",
    "pedido_id": "identificador",
    "fecha_evento": "fecha",
    "estado_evento": "categorica",
    "ciudad_destino": "categorica",
    "transportadora": "categorica",
    "incidencia": "categorica",
    "tiempo_etapa_horas": "numerica",
    "costo_envio": "monetaria",
}
print("\nRevision de tipos - df_logistica")
display(quality.revisar_tipos(df_logistica, tipos_esperados_logistica))

## Parte 5. Estadísticos descriptivos y resúmenes

Análisis descriptivo de cada fuente **por separado**, sin integrarlas todavía.

In [ ]:
# 5.1 Variables numericas: count, mean, std, min, p25, mediana, p75, max
cols_numericas_ventas = ["cantidad", "precio_unitario", "valor_bruto", "valor_descuento", "valor_neto", "calificacion_cliente"]
print("Resumen numerico - df_ventas")
display(stats.resumen_numerico(df_ventas, cols_numericas_ventas))

cols_numericas_logistica = ["tiempo_etapa_horas", "costo_envio"]
print("\nResumen numerico - df_logistica")
display(stats.resumen_numerico(df_logistica, cols_numericas_logistica))

In [ ]:
# 5.2 Variables categoricas + resumenes especificos solicitados para df_ventas
resultados_ventas = stats.resumen_ventas(df_ventas)
for titulo, tabla in resultados_ventas.items():
    print(f"\n--- {titulo} ---")
    display(tabla)

In [ ]:
# Resumenes especificos solicitados para df_logistica
resultados_logistica = stats.resumen_logistica(df_logistica)
for titulo, tabla in resultados_logistica.items():
    print(f"\n--- {titulo} ---")
    display(tabla)

## Parte 6. Preguntas de negocio (conocimiento previo)

10 preguntas resueltas con `pandas` directamente sobre los datos **originales** (sin integrar), usando las funciones de `kaismart_etl/business_questions.py`.

In [ ]:
print("1. Valor neto total de ventas por ciudad")
print(bq.q1_ciudad_mayor_valor_neto(df_ventas))

print("\n2. Numero de ventas por canal")
print(bq.q2_canal_mas_ventas(df_ventas))

print("\n3. Cantidad total vendida por categoria")
print(bq.q3_categoria_mas_vendida_cantidad(df_ventas))

print("\n4. Top 10 clientes por valor neto acumulado")
print(bq.q4_top10_clientes_valor_neto(df_ventas))

print("\n5. Numero de ventas por mes")
print(bq.q5_ventas_por_mes(df_ventas))

In [ ]:
print("6. Transportadora con mas incidencias reportadas")
print(bq.q6_transportadora_mas_incidencias(df_logistica))

print("\n7. Estado de evento logistico mas frecuente")
print(bq.q7_estado_evento_mas_frecuente(df_logistica))

print("\n8. Tiempo promedio de etapa (horas) por transportadora")
print(bq.q8_tiempo_promedio_por_transportadora(df_logistica))

print("\n9. Costo de envio promedio por ciudad de destino")
print(bq.q9_costo_envio_promedio_por_ciudad(df_logistica))

print("\n10. % de ventas con calificacion de cliente y promedio")
print(bq.q10_pct_ventas_con_calificacion(df_ventas))

## Cierre del EDA

Los problemas de calidad detectados y cuantificados en este notebook (nulos, duplicados, categorías inconsistentes, tipos de dato incorrectos, valores fuera de rango) se resuelven de forma explícita y documentada en el notebook [`02_transformacion_medallion.ipynb`](02_transformacion_medallion.ipynb), que implementa las capas **Bronze → Silver → Gold** y produce `df_ventas_transformado` y `df_logistica_transformado`.